<!-- # <span style="color:red">UNDER CONSTRUCTION!!!!</span> -->

# Spoken Language Processing - Instituto Superior Técnico
### Laboratory Assignment 2 - Automatic Age Estimation Challenge
<!--[image](imgs/lab2_slp_banner.png)-->
<img src="imgs/lab2_slp_banner.png" alt="drawing" width="400"/>

# WEEK 2 - Using pre-trained models


During this week, students will implement two modern systems for age regression based on:
- speaker representations (utterance-based) obtained with an x-vector model (this notebook);
- speech representations (frame-based) obtained with a self-supervised learning (SSL) pre-trained model (`lab2_ssl.ipynb` notebook).

In both cases, students are encouraged to explore different feature configurations and alternative downstream models.

## Before starting

Let's import some modules and make some definitions. 

Like in the previous Notebooks, you need to upload pf_tools.py and requirements.txt if you are working on Google Colab. Otherwise, you should skip or delete the following code cell:

In [1]:
!pip install -r requirements.txt # Run this cell if you are using Google colab

ERROR: Invalid requirement: '#'
You should consider upgrading via the 'C:\Users\dinis\OneDrive\Desktop\Processamento_da_fala\processamento_da_fala\Scripts\python.exe -m pip install --upgrade pip' command.


**WARNING from professors** We changed the pf_tools.py script for this second week. Be sure to update (the new one is compatible with Week 1 lab)

In [2]:
import os
import csv
import pickle
import numpy as np
import librosa
import torch

from pf_tools import CheckThisCell, SLPdata
from speechbrain.inference.classifiers import EncoderClassifier
from speechbrain.utils.data_utils import split_path
from sklearn.svm import LinearSVC, SVR
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import matplotlib.pyplot as plt


GENDER_CLASSES = ('F',  'M')
GEN2ID = {'F':0, 'M':1}
ID2GEN = dict((GEN2ID[k],k)for k in GEN2ID)

c:\Users\dinis\OneDrive\Desktop\Processamento_da_fala\processamento_da_fala\lib\site-packages\speechbrain\utils\torch_audio_backend.py:64: UserWarning: torchaudio._backend.list_audio_backends has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be removed from the 2.9 release. 
  available_backends = torchaudio.list_audio_backends()


Like in the previous Notebooks, you need to mount Google drive if you are working on Google Colab. Otherwise, you should skip or delete the following code cell:


Like in week1, the audio data is expected to be in a folder with the following format:

```
lab2_data/
├── train/
│   └── wav/
│       └──wav files
│   └── info.csv
│
└── train_small/
    └── wav/
        └──wav files
    └── info.csv
...
```

You must already have this from the previous week, so you can set-up your data directory:

In [3]:

# raise CheckThisCell ## <---- Remove this after completing/checking this cell

CWD = os.getcwd()
DATADIR = f'{CWD}/lab2_data/' # <--- Change this variable to your working directory containig the SLP data
if not os.path.isdir(DATADIR):
    os.mkdir(DATADIR)
    print(f'WARNING: Your data is not in the folder {DATADIR}')

os.chdir(CWD)
print(f'Your SLP data should be in this folder {DATADIR}')


Your SLP data should be in this folder c:\Users\dinis\OneDrive\Desktop\Processamento_da_fala\Lab_2/lab2_data/


If you need to download again the data, you can run the following cell:

In [3]:
# raise CheckThisCell

os.chdir(DATADIR)

# download train
!wget http://groups.tecnico.ulisboa.pt/speechproc/pf26/lab2/train.tgz
!tar -xzvf train.tgz

#download train100
!wget http://groups.tecnico.ulisboa.pt/speechproc/pf26/lab2/train_small.tgz
!tar -xzvf train_small.tgz

#download dev
!wget http://groups.tecnico.ulisboa.pt/speechproc/pf26/lab2/dev.tgz
!tar -xzvf dev.tgz

#download evl
!wget http://groups.tecnico.ulisboa.pt/speechproc/pf26/lab2/evl.tgz
!tar -xzvf evl.tgz

os.chdir(CWD)

--2026-05-19 14:50:10--  http://groups.tecnico.ulisboa.pt/speechproc/pf26/lab2/train.tgz
Resolving groups.tecnico.ulisboa.pt (groups.tecnico.ulisboa.pt)... 2001:690:2100:4::d301:2ae9, 193.136.128.24
Connecting to groups.tecnico.ulisboa.pt (groups.tecnico.ulisboa.pt)|2001:690:2100:4::d301:2ae9|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2062976726 (1.9G) [application/x-gzip]
Saving to: 'train.tgz.3'

     0K .......... .......... .......... .......... ..........  0%  837K 40m8s
    50K .......... .......... .......... .......... ..........  0%  360K 66m39s
   100K .......... .......... .......... .......... ..........  0%  605K 62m57s
   150K .......... .......... .......... .......... ..........  0%  707K 59m4s
   200K .......... .......... .......... .......... ..........  0% 2.83M 49m35s
   250K .......... .......... .......... .......... ..........  0%  628K 50m13s
   300K .......... .......... .......... .......... ..........  0% 2.85M 44m42s
   350K ..

## Using pre-trained speaker embeddings (x-vectors)

The goal of this part of the lab is to become familiar with and show how to use pre-trained speaker embedings (a.k.a. x-vectors) for speech classification/regressions tasks.

There exist plenty of resources and pre-trained models that can be  useful for our task. In particular, x-vectors are the current state of the art approach to obtain speech embeddings that characterize very efficiently speaker or language, among others. X-vectors are neural models typically trained for speaker identification in a supervised way, but also in some cases for other related tasks. Once trained, they can be used to obtain a single embedding vector of fixed dimension for each audio input. This vector corresponds to the activations of one of the layers after the pooling layer.

The following are examples of x-vector models available in the `speechbrain` module:

- `speechbrain/spkrec-xvect-voxceleb`: same with a different architecture: https://huggingface.co/speechbrain/spkrec-xvect-voxceleb

- `speechbrain/spkrec-ecapa-voxceleb`: trained using a large speaker corpus for speaker verification: https://huggingface.co/speechbrain/spkrec-ecapa-voxceleb



The following code cell shows how to import one of those models to obtain an embedding vector:

In [4]:
# raise CheckThisCell ## <---- Remove this after completing/checking this cell 

# Instantiate the model. If you don't have GPU available, run this line
xvector_model = EncoderClassifier.from_hparams(source="speechbrain/spkrec-xvect-voxceleb", savedir=f"{CWD}/tmp")

# If you have GPU available, run this line instead
# xvector_model = EncoderClassifier.from_hparams(source="speechbrain/spkrec-xvect-voxceleb", savedir=f"{CWD}/tmp", run_opts={"device":"cuda"})

signal = xvector_model.load_audio(f'{DATADIR}/train_small/wav/00834c0e904d40eda496e55010acebc5.wav')
emb =  xvector_model.encode_batch(signal)

print(type(emb), emb.shape)

c:\Users\dinis\OneDrive\Desktop\Processamento_da_fala\processamento_da_fala\lib\site-packages\speechbrain\utils\torch_audio_backend.py:64: UserWarning: torchaudio._backend.list_audio_backends has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be removed from the 2.9 release. 
  available_backends = torchaudio.list_audio_backends()
c:\Users\dinis\OneDrive\Desktop\Processamento_da_fala\processamento_da_fala\lib\site-packages\speechbrain\utils\parameter_transfer.py:234: UserWarning: Requested Pretrainer collection using symlinks on Windows. This might not work; see `LocalStrategy` documentation. Consider unsetting `collect_in` in Pretrainer to avoid symlinking altogether.
  warnings.warn(


<class 'torch.Tensor'> torch.Size([1, 1, 512])


These (very informative) embedding vectors can be used to train simple models for several speech classification tasks, achieving excellent results. In particular, in this lab assignment, we will train a simple Support Vector Regression (SVR) on top of these x-vectors.


Student groups will be graded depending on their ability to explore different feature configurations and model alternatives/configurations.

### 1. Extracting x-vectors for the SLP datasets

Just like in Part1, we will code the feature transformation to process all data and obtain x-vectors. In this case, the function should receive as arguments the audio filename and an instance of `EncoderClassifier` (the x-vector model) and return the numpy array with the features. You must complete the following code using the previous example:

### 1.1 Audio preprocessing options

Before extracting x-vectors or ECAPA embeddings, we can choose to clean the waveform. The goal is to make the frozen speaker encoder focus on speech frames instead of silence or recording-level variability.

In this subsection we will expose four selectable preprocessing modes:

1. **Trim only**: remove leading and trailing silence with `librosa.effects.trim`.
2. **Trim + RMS normalization**: trim the waveform and then normalize its loudness to a fixed RMS target.
3. **Speech-only VAD**: keep only regions detected as speech by a pre-trained SpeechBrain VAD model.
4. **Speech-only VAD + RMS normalization**: keep only speech and then normalize its loudness.

The rest of the notebook stays the same. The only thing you need to do is run **one** of the choice cells below before generating the cached features.


The next helper cell defines the preprocessing functions and a global selector used by `extract_xvec(...)`. If you do not run any of the four choice cells after it, the notebook keeps the original **raw-audio** behaviour.


In [ ]:
# improvements pls

from pathlib import Path
from speechbrain.inference.VAD import VAD
from speechbrain.utils.fetching import LocalStrategy

PREPROCESS_SAMPLE_RATE = 16000
TARGET_RMS = 0.1
vad_model = None

audio_preprocess_id = 'raw'
AUDIO_PREPROCESS_LABELS = {
    'raw': 'raw audio (no extra preprocessing)',
    'trim': 'trim leading/trailing silence',
    'trim_rms': 'trim silence + RMS normalization',
    'vad': 'speech-only VAD',
    'vad_rms': 'speech-only VAD + RMS normalization',
}


def make_cached_transform_id(base_transform_id):
    if audio_preprocess_id == 'raw':
        return base_transform_id
    return f'{base_transform_id}__{audio_preprocess_id}'


def set_audio_preprocessing(mode):
    global audio_preprocess_id
    if mode not in AUDIO_PREPROCESS_LABELS:
        raise ValueError(f'Unknown audio preprocessing mode: {mode}')
    audio_preprocess_id = mode
    print(f'Using audio preprocessing: {AUDIO_PREPROCESS_LABELS[audio_preprocess_id]}')


def rms_normalize_waveform(waveform, target_rms=TARGET_RMS, eps=1e-8):
    if waveform.size == 0:
        return waveform
    rms = np.sqrt(np.mean(np.square(waveform), dtype=np.float64) + eps)
    if rms < eps:
        return waveform
    normalized = waveform * (target_rms / rms)
    peak = np.max(np.abs(normalized)) if normalized.size else 0.0
    if peak > 1.0:
        normalized = normalized / peak
    return normalized.astype(np.float32)


def trim_waveform(waveform, top_db=30):
    trimmed, _ = librosa.effects.trim(waveform, top_db=top_db)
    if trimmed.size == 0:
        return waveform.astype(np.float32)
    return trimmed.astype(np.float32)


def load_vad_model():
    global vad_model
    if vad_model is None:
        vad_model = VAD.from_hparams(
            source='speechbrain/vad-crdnn-libriparty',
            savedir=f'{CWD}/tmp/vad-crdnn-libriparty',
            local_strategy=LocalStrategy.COPY,
        )
    return vad_model


def apply_speechbrain_vad(filename, waveform, sample_rate=PREPROCESS_SAMPLE_RATE):
    vad = load_vad_model()
    boundaries = vad.get_speech_segments(
        filename,
        apply_energy_VAD=True,
        close_th=0.15,
        len_th=0.15,
    )

    speech_segments = []
    for start_sec, end_sec in boundaries.cpu().tolist():
        start_sample = max(0, int(round(start_sec * sample_rate)))
        end_sample = min(len(waveform), int(round(end_sec * sample_rate)))
        if end_sample > start_sample:
            speech_segments.append(waveform[start_sample:end_sample])

    if not speech_segments:
        return waveform.astype(np.float32)

    return np.concatenate(speech_segments).astype(np.float32)


def preprocess_waveform_for_embedding(filename):
    filename = Path(filename).as_posix()
    waveform, _ = librosa.load(filename, sr=PREPROCESS_SAMPLE_RATE, mono=True)
    waveform = np.asarray(waveform, dtype=np.float32)
    original_waveform = waveform.copy()

    if audio_preprocess_id in ('trim', 'trim_rms'):
        waveform = trim_waveform(waveform)
    elif audio_preprocess_id in ('vad', 'vad_rms'):
        waveform = apply_speechbrain_vad(filename, waveform, sample_rate=PREPROCESS_SAMPLE_RATE)

    if waveform.size == 0:
        waveform = original_waveform

    if audio_preprocess_id in ('trim_rms', 'vad_rms'):
        waveform = rms_normalize_waveform(waveform)

    return filename, waveform


print(f'Default audio preprocessing: {AUDIO_PREPROCESS_LABELS[audio_preprocess_id]}')


**Option 1 — Trim only**

This removes only the leading and trailing silence. It is the cheapest intervention and often a good first test when recordings include long quiet margins.


In [ ]:
# improvements pls
set_audio_preprocessing('trim')


**Option 2 — Trim + RMS normalization**

This first removes the silent margins and then rescales the waveform so that all files have a more comparable average loudness. This can make the encoder less sensitive to recording-level gain differences.


In [ ]:
# improvements pls
set_audio_preprocessing('trim_rms')


**Option 3 — Speech-only VAD**

This uses a pre-trained SpeechBrain VAD model to keep only detected speech regions. It is more aggressive than simple trimming because it can remove internal pauses and non-speech stretches as well.


In [ ]:
# improvements pls
set_audio_preprocessing('vad')


**Option 4 — Speech-only VAD + RMS normalization**

This keeps only speech according to the VAD model and then applies RMS normalization. Among the four options, this is the most interventionist waveform cleanup strategy.


In [ ]:
# improvements pls
set_audio_preprocessing('vad_rms')


In [ ]:
# raise CheckThisCell ## <---- Remove this after completing/checking this cell

# This function receives a filename and one encoder model
# (for instance, xvector_model object in previous cell example) and returns
# the extracted x-vectors as a numpy array of shape (1, embedding_dimension).
# The selected audio preprocessing mode is applied before the frozen encoder.
def extract_xvec(filename, emb_model):
    filename, waveform = preprocess_waveform_for_embedding(filename)
    signal = torch.tensor(waveform, dtype=torch.float32)

    if hasattr(emb_model, 'audio_normalizer') and emb_model.audio_normalizer is not None:
        signal = emb_model.audio_normalizer(signal, PREPROCESS_SAMPLE_RATE)

    emb = emb_model.encode_batch(signal)
    embedding = emb.squeeze(1).detach().cpu().numpy()
    return embedding

# This must return a numpy array of shape (1,D).
# D can vary depending on the model you are using, but it is usually 192 or 512.
emb = extract_xvec(f'{DATADIR}/train_small/wav/00834c0e904d40eda496e55010acebc5.wav', xvector_model)

# Check that this shape and type.
# It should be: (1, 512) <class 'numpy.ndarray'> or (1, 192) <class 'numpy.ndarray'>
print(emb.shape, type(emb))


Let's generate the x-vectors for all our data sets using the SLP class and store in disk. Like in Part 1, we can keep different transformation configurations in a dictionary for later usage.  

Let's define first our configurations (you can try to different x-vector models, the propsed ones or even other that you may find in huggingface):

In [6]:

# raise CheckThisCell ## <---- Remove this after completing/checking this cell

# transform = {
#                 'spkrec-ecapa-voxceleb' : # <--- You chan look for different models in speechbrain
#                 {
#                     'audio_transform':
#                         lambda x : extract_xvec(x,
#                             emb_model = EncoderClassifier.from_hparams(
#                                         source="speechbrain/spkrec-ecapa-voxceleb", # <--- You can look for different models in speechbrain
#                                         savedir=f"{CWD}/tmp/spkrec-ecapa-voxceleb"
#                                         )
#                             ), ## <--- You need to modify this here
#                     'chunk_transform': None,
#                     'chunk_size': 0,
#                     'chunk_hop':0
#                 }
#             }

# transform['spkrec-xvect-voxceleb'] = {
#                     'audio_transform':
#                         lambda x : extract_xvec(x,
#                             emb_model = EncoderClassifier.from_hparams(
#                                         source="speechbrain/spkrec-xvect-voxceleb", # <--- You can look for different models in speechbrain
#                                         savedir=f"{CWD}/tmp/spkrec-xvect-voxceleb"
#                                         )
#                             ), ## <--- You need to modify this here
#                     'chunk_transform': None,
#                     'chunk_size': 0,
#                     'chunk_hop':0
#                 }

# Build each model ONCE
from speechbrain.utils.fetching import LocalStrategy

ecapa_model = EncoderClassifier.from_hparams(
    source="speechbrain/spkrec-ecapa-voxceleb",
    savedir=f"{CWD}/tmp/spkrec-ecapa-voxceleb",
    local_strategy=LocalStrategy.COPY,
)

xvect_model = EncoderClassifier.from_hparams(
    source="speechbrain/spkrec-xvect-voxceleb",
    savedir=f"{CWD}/tmp/spkrec-xvect-voxceleb",
    local_strategy=LocalStrategy.COPY,
)


transform = {
    'spkrec-ecapa-voxceleb': {
        'audio_transform': lambda x: extract_xvec(x, emb_model=ecapa_model),
        'chunk_transform': None,
        'chunk_size': 0,
        'chunk_hop': 0,
    },
    'spkrec-xvect-voxceleb': {
        'audio_transform': lambda x: extract_xvec(x, emb_model=xvect_model),
        'chunk_transform': None,
        'chunk_size': 0,
        'chunk_hop': 0,
    },
}

And now let's do feature extraction. Be patient because this process can be a bit slow depending on the resources of your machine (train_small without GPU should take around 5min on google colab):

In [ ]:
# raise CheckThisCell ## <---- Remove this after completing/checking this cell

# Download and feature extract
trainset = 'train_small'

FEATURE_DIMS = {
    'spkrec-xvect-voxceleb': 512,
    'spkrec-ecapa-voxceleb': 192,
}


def load_feature_partitions(selected_transform_id):
    global transform_id, base_transform_id, feat_dim, slp_partitions

    base_transform_id = selected_transform_id
    transform_id = make_cached_transform_id(base_transform_id)
    feat_dim = FEATURE_DIMS[base_transform_id]
    slp_partitions = {}

    transform_cfg = transform[base_transform_id]

    # for partition in ('train', 'train_small', 'dev', 'evl'):
    for partition in (trainset, 'dev', 'evl'):
        slp_partitions[partition] = SLPdata(
            DATADIR,
            partition,
            transform_id=transform_id,
            audio_transform=transform_cfg['audio_transform'],
            chunk_transform=transform_cfg['chunk_transform'],
            chunk_size=transform_cfg['chunk_size'],
            chunk_hop=transform_cfg['chunk_hop'],
        )

    print(f'Using encoder: {base_transform_id}')
    print(f'Using audio preprocessing: {AUDIO_PREPROCESS_LABELS[audio_preprocess_id]}')
    print(f'Feature cache id: {transform_id}')
    print(f'Feature dimension: {feat_dim}')



In [ ]:
# Run this cell to use x-vector embeddings in the rest of the notebook.
load_feature_partitions('spkrec-xvect-voxceleb')


In [ ]:
# Run this cell to use ECAPA embeddings in the rest of the notebook.
load_feature_partitions('spkrec-ecapa-voxceleb')


### 2. Training an SVR model

Our first attempt of age regression system based on x-vectors will be a simple SVR model like in the `openSMILE` baseline, but in this case we will be using x-vectors as features.

First, we will use the SLP data instances to store the x-vectors, the labels and file identifiers in numpy arrays:

In [9]:
from pf_tools import prepare_slp_data

#   Concatenate all data and labels
#   Each row corresponds to a file
#   We store the data, labels and file identifiers of each partition in dictionaries
#     with the partition name as key


gender_label_pos = 0
age_label_pos = 1

data, labels_gender, labels_age, fileids = {}, {}, {}, {}
# for partition in ('train', 'train_small', 'dev', 'evl'):
for partition in ('train_small', 'dev', 'evl'):
    data_and_labels = prepare_slp_data(slp_partitions[partition])
    print(f'Partition: {partition}')
    print(f'Number of samples: {data_and_labels["data"].shape[0]}')
    print(f'Number of features: {data_and_labels["data"].shape[1]}')
    print(f'Number of labels: {len(np.unique(data_and_labels["label"][:,gender_label_pos]))}')
    print(f'Number of identifiers (samples): {len(np.unique(data_and_labels["identifiers"]))}')
    print('---')
    data[partition] = data_and_labels['data']
    labels_gender[partition] = data_and_labels['label'][:,gender_label_pos]
    labels_age[partition] = data_and_labels['label'][:,age_label_pos]
    fileids[partition] = data_and_labels['identifiers']



Partition: train_small
Number of samples: 647
Number of features: 512
Number of labels: 2
Number of identifiers (samples): 647
---
Partition: dev
Number of samples: 117
Number of features: 512
Number of labels: 2
Number of identifiers (samples): 117
---
Partition: evl
Number of samples: 145
Number of features: 512
Number of labels: 1
Number of identifiers (samples): 145
---


Now, we will use `sklearn` Support Vector Regression (SVR) to:
1. Train our regressor and save it for later use.
2. Predict on the dev and evl partitions and save the results

In [10]:
from sklearn.svm import SVR
from pf_tools import save_model

trainset = 'train_small'

# Train a SVR
model = SVR(kernel='linear') ### <---- a linear SVR
model.fit(data[trainset], labels_age[trainset])  ## <---- train MODEL

model_id = save_model(model, f'svr_{transform_id}', f'{DATADIR}/{trainset}/models/')
print(f'Model {model_id} saved in {DATADIR}/{trainset}/models/')

# Predict the dev and evl sets
dev_results = model.predict(data['dev']) #  Predict dev
filename = f'{DATADIR}/{trainset}/models/{model_id}/dev.pkl'
pickle.dump({'hyp':dev_results, 'fileids':fileids['dev']}, open(filename, 'wb'))

evl_results = model.predict(data['evl']) #  Predict evl
filename = f'{DATADIR}/{trainset}/models/{model_id}/evl.pkl'
pickle.dump({'hyp':evl_results, 'fileids':fileids['evl']}, open(filename, 'wb'))


Model saved to c:\Users\dinis\OneDrive\Desktop\Processamento_da_fala\Lab_2/lab2_data//train_small/models//svr_spkrec-xvect-voxceleb_2026-05-19_18-47-07/model.pkl
Model svr_spkrec-xvect-voxceleb_2026-05-19_18-47-07 saved in c:\Users\dinis\OneDrive\Desktop\Processamento_da_fala\Lab_2/lab2_data//train_small/models/


It should be extremely easy to experiment other models provided in the `sklearn` module, including SVMs with other kernels, Random Forests, etc.


#### 2.1 Analyze results on the dev set and prepare your submission file

Let's check our performance on the dev set:

In [11]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

ref, hyp = labels_age['dev'], dev_results

print(f'Mean Absolute Error: {mean_absolute_error(ref, hyp):.2f}')
print(f'Mean Squared Error: {mean_squared_error(ref, hyp):.2f}')

Mean Absolute Error: 7.61
Mean Squared Error: 90.25


You should obtain a mean absolute error around 7.6 (it will depend on the xvector model chosen). 

Now, let's generate the final prediction file and make a submission to the  [Kaggle competition](https://www.kaggle.com/t/8d80747e0c474688a83024aabdfe1ab0):

In [ ]:
from pf_tools import create_submission_file

students_group = '07' # <--- CHANGE THIS ACCORDINGLY

model_id_short = {
    'spkrec-xvect-voxceleb': 'svr_spkrec-xvect',
    'spkrec-ecapa-voxceleb': 'svr_spkrec-ecapa',
}.get(transform_id, f'svr_{transform_id}')

results_path = f'{DATADIR}/{trainset}/models/{model_id}/'
filename = f'{CWD}/g{students_group}_{trainset}_{model_id_short}.csv' # <--- CHANGE THIS ACCORDINGLY

create_submission_file(results_path, filename)


At this point, you can explore different x-vector model configurations for feature extraction and alternative models to the SVR.

### 2.2 Comparing x-vector and ECAPA with fusion strategies

Instead of choosing only one frozen speaker encoder, we can compare two encoders under the same downstream regression setup. In this subsection we will:

1. Load both `spkrec-xvect-voxceleb` and `spkrec-ecapa-voxceleb` features.
2. Align the partitions by `fileid` so that all comparisons are made on the same utterances.
3. Train one regressor per encoder and average their predictions (**late fusion**).
4. Concatenate the embeddings into a single `704`-dimensional vector (**feature-level concatenation**).

The point is to isolate whether the gain comes from the representation itself or from combining complementary encoders.


In [ ]:
# improvements pls

from sklearn.base import clone
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from pf_tools import prepare_slp_data, save_model


def prepare_feature_bundle(selected_transform_id):
    age_label_pos = 1
    bundle = {}

    for partition in (trainset, 'dev', 'evl'):
        cached_transform_id = make_cached_transform_id(selected_transform_id)
        slp_partition = SLPdata(
            DATADIR,
            partition,
            transform_id=cached_transform_id,
            audio_transform=transform[selected_transform_id]['audio_transform'],
            chunk_transform=transform[selected_transform_id]['chunk_transform'],
            chunk_size=transform[selected_transform_id]['chunk_size'],
            chunk_hop=transform[selected_transform_id]['chunk_hop'],
        )
        prepared = prepare_slp_data(slp_partition)
        partition_bundle = {
            'X': prepared['data'],
            'fileids': np.asarray(prepared['identifiers']),
        }
        if partition != 'evl':
            partition_bundle['y'] = prepared['label'][:, age_label_pos].astype(np.float32)
        else:
            partition_bundle['y'] = None
        bundle[partition] = partition_bundle

    return bundle


def align_encoder_bundles(primary_bundle, secondary_bundle):
    aligned = {}

    for partition in (trainset, 'dev', 'evl'):
        primary_ids = np.asarray(primary_bundle[partition]['fileids'])
        secondary_ids = np.asarray(secondary_bundle[partition]['fileids'])
        secondary_index = {fid: idx for idx, fid in enumerate(secondary_ids)}

        missing = [fid for fid in primary_ids if fid not in secondary_index]
        if missing:
            raise ValueError(
                f'Missing fileids while aligning {partition}: {missing[:5]}'
            )

        order = np.array([secondary_index[fid] for fid in primary_ids])
        aligned_partition = {
            'fileids': primary_ids,
            'xvec': np.asarray(primary_bundle[partition]['X']),
            'ecapa': np.asarray(secondary_bundle[partition]['X'])[order],
        }

        if partition != 'evl':
            primary_y = np.asarray(primary_bundle[partition]['y'], dtype=np.float32)
            secondary_y = np.asarray(secondary_bundle[partition]['y'], dtype=np.float32)[order]
            if not np.allclose(primary_y, secondary_y):
                raise ValueError(f'Age labels do not match after alignment for {partition}.')
            aligned_partition['y'] = primary_y
        else:
            aligned_partition['y'] = None

        aligned_partition['concat'] = np.concatenate(
            [aligned_partition['xvec'], aligned_partition['ecapa']], axis=1
        )
        aligned[partition] = aligned_partition

    return aligned


fusion_regressors = {
    'ridge': {
        'name': 'Ridge',
        'estimator': Pipeline([
            ('scaler', StandardScaler()),
            ('ridge', Ridge(alpha=1.0)),
        ]),
    },
    'linear_svr': {
        'name': 'Linear SVR',
        'estimator': Pipeline([
            ('scaler', StandardScaler()),
            ('svr', SVR(kernel='linear', C=1.0, epsilon=0.1)),
        ]),
    },
}


The next cell prepares both encoder feature sets. If one feature directory is missing, `SLPdata(...)` will extract it automatically. We then align the two bundles by `fileid`, which guarantees that the late-fusion averaging and the concatenation use exactly the same utterances in the same order.


In [ ]:
# improvements pls

xvec_bundle = prepare_feature_bundle('spkrec-xvect-voxceleb')
ecapa_bundle = prepare_feature_bundle('spkrec-ecapa-voxceleb')
dual_encoder_data = align_encoder_bundles(xvec_bundle, ecapa_bundle)

for partition in (trainset, 'dev', 'evl'):
    part = dual_encoder_data[partition]
    print(f'Partition: {partition}')
    print(f'  x-vector shape:      {part["xvec"].shape}')
    print(f'  ECAPA shape:         {part["ecapa"].shape}')
    print(f'  concatenated shape:  {part["concat"].shape}')
    print(f'  number of fileids:   {len(part["fileids"])}')
    print('---')


We now compare four setups for each downstream regressor family:

- **x-vector only**: one model trained only on the 512-dimensional x-vector.
- **ECAPA only**: one model trained only on the 192-dimensional ECAPA embedding.
- **Late fusion**: train one model per encoder, predict separately, and average their outputs.
- **Concatenation**: join the two embeddings into a single 704-dimensional vector and train one model on top.

We keep the downstream family fixed within each block so that the comparison remains fair.


In [ ]:
# improvements pls

fusion_results = []


def register_result(reg_name, variant_key, display_name, artifact, dev_pred, evl_pred):
    result = {
        'reg_name': reg_name,
        'variant_key': variant_key,
        'display_name': display_name,
        'artifact': artifact,
        'dev_pred': dev_pred,
        'evl_pred': evl_pred,
        'dev_mae': mean_absolute_error(dual_encoder_data['dev']['y'], dev_pred),
        'dev_mse': mean_squared_error(dual_encoder_data['dev']['y'], dev_pred),
        'slug': f'{reg_name.lower().replace(" ", "_")}_{variant_key}',
    }
    fusion_results.append(result)


X_train_xvec = dual_encoder_data[trainset]['xvec']
X_train_ecapa = dual_encoder_data[trainset]['ecapa']
X_train_concat = dual_encoder_data[trainset]['concat']
y_train_fusion = dual_encoder_data[trainset]['y']

X_dev_xvec = dual_encoder_data['dev']['xvec']
X_dev_ecapa = dual_encoder_data['dev']['ecapa']
X_dev_concat = dual_encoder_data['dev']['concat']

X_evl_xvec = dual_encoder_data['evl']['xvec']
X_evl_ecapa = dual_encoder_data['evl']['ecapa']
X_evl_concat = dual_encoder_data['evl']['concat']

for reg_key, reg_cfg in fusion_regressors.items():
    reg_name = reg_cfg['name']
    prototype = reg_cfg['estimator']

    xvec_model = clone(prototype)
    xvec_model.fit(X_train_xvec, y_train_fusion)
    xvec_dev_pred = xvec_model.predict(X_dev_xvec)
    xvec_evl_pred = xvec_model.predict(X_evl_xvec)
    register_result(reg_name, 'xvec', f'{reg_name} | x-vector only', xvec_model, xvec_dev_pred, xvec_evl_pred)

    ecapa_model = clone(prototype)
    ecapa_model.fit(X_train_ecapa, y_train_fusion)
    ecapa_dev_pred = ecapa_model.predict(X_dev_ecapa)
    ecapa_evl_pred = ecapa_model.predict(X_evl_ecapa)
    register_result(reg_name, 'ecapa', f'{reg_name} | ECAPA only', ecapa_model, ecapa_dev_pred, ecapa_evl_pred)

    late_dev_pred = 0.5 * (xvec_dev_pred + ecapa_dev_pred)
    late_evl_pred = 0.5 * (xvec_evl_pred + ecapa_evl_pred)
    late_artifact = {
        'type': 'late_fusion',
        'xvec_model': xvec_model,
        'ecapa_model': ecapa_model,
        'xvec_transform_id': make_cached_transform_id('spkrec-xvect-voxceleb'),
        'ecapa_transform_id': make_cached_transform_id('spkrec-ecapa-voxceleb'),
        'audio_preprocess_id': audio_preprocess_id,
    }
    register_result(reg_name, 'late_fusion', f'{reg_name} | late fusion', late_artifact, late_dev_pred, late_evl_pred)

    concat_model = clone(prototype)
    concat_model.fit(X_train_concat, y_train_fusion)
    concat_dev_pred = concat_model.predict(X_dev_concat)
    concat_evl_pred = concat_model.predict(X_evl_concat)
    register_result(reg_name, 'concatenation', f'{reg_name} | concatenation', concat_model, concat_dev_pred, concat_evl_pred)

fusion_results = sorted(fusion_results, key=lambda result: result['dev_mae'])

print('=== Fusion comparison on the dev set ===')
print(f'{"Experiment":<32}{"MAE":>10}{"MSE":>12}')
for result in fusion_results:
    print(f'{result["display_name"]:<32}{result["dev_mae"]:>10.3f}{result["dev_mse"]:>12.3f}')

best_overall_result = fusion_results[0]
fusion_only_results = [
    result for result in fusion_results
    if result['variant_key'] in ('late_fusion', 'concatenation')
]
best_fusion_result = fusion_only_results[0]

print() 
print('Best overall experiment:', best_overall_result['display_name'])
print('Best fusion-style experiment:', best_fusion_result['display_name'])



If one of the fusion-style systems beats the single-encoder baselines, the next cells save its predictions in the same `dev.pkl` / `evl.pkl` format used in the rest of the notebook. That makes it easy to generate a submission file without changing the earlier sections.


In [ ]:
# improvements pls

result_to_save = best_fusion_result
fusion_model_tag = f'xvec_ecapa_{result_to_save["slug"]}'

model_id = save_model(
    result_to_save['artifact'],
    fusion_model_tag,
    f'{DATADIR}/{trainset}/models/',
)
model_id_short = fusion_model_tag

filename = f'{DATADIR}/{trainset}/models/{model_id}/dev.pkl'
pickle.dump({'hyp': result_to_save['dev_pred'], 'fileids': dual_encoder_data['dev']['fileids']}, open(filename, 'wb'))

filename = f'{DATADIR}/{trainset}/models/{model_id}/evl.pkl'
pickle.dump({'hyp': result_to_save['evl_pred'], 'fileids': dual_encoder_data['evl']['fileids']}, open(filename, 'wb'))

print(f'Saved best fusion-style experiment: {result_to_save["display_name"]}')
print(f'Model identifier: {model_id}')
print(f'Dev/Evl prediction files saved to: {DATADIR}/{trainset}/models/{model_id}/')


The final cell below mirrors the earlier submission workflow, but it uses the best fusion-style experiment selected in this subsection.


In [ ]:
# improvements pls

from pf_tools import create_submission_file

students_group = '07' # <--- CHANGE THIS ACCORDINGLY

results_path = f'{DATADIR}/{trainset}/models/{model_id}/'
filename = f'{CWD}/g{students_group}_{trainset}_{model_id_short}.csv' # <--- CHANGE THIS ACCORDINGLY

create_submission_file(results_path, filename)


### 3. Training a neural network model

As an alternative to the SVR, we will explore simple neural models on top of x-vector features.

We will need to define the size of the feature vector that will be used as input to the neural network:

In [ ]:
# raise CheckThisCell ## <---- Remove this after completing/checking this cell
# feat_dim is now set automatically by the x-vector / ECAPA selection cell above.
feat_dim = data[trainset].shape[1]
print(f'Current transform_id: {transform_id}')
print(f'Feature dimension inferred from prepared data: {feat_dim}')


And a simple neural model architecture (you can change this):

In [16]:
import torch
from torch import nn

# Define a simple linear neural model
class LinearRegressor(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super(LinearRegressor, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, x):
        return self.model(x)

model = LinearRegressor(input_dim=feat_dim, hidden_dim=50)

We will use a simple `train_nn` function included in the `pf_tools` script that will permit training the model using backpropagation. Students are encouraged to explore this function and, eventually, to modify it to experiment alternative training strategies, parameters, etc.

In [17]:
from pf_tools import train_nn, predict_nn, save_model

train_nn(model, slp_partitions[trainset], slp_partitions['dev'], batch_size=16, epochs=1000, lr=0.001,)

model_id = save_model(model, f'nnet_{transform_id}', f'{DATADIR}/{trainset}/models/')
print(f'Model {model_id} saved in {DATADIR}/{trainset}/models/')

Epoch 10/1000, Train MSE Loss: 108.9194
Dev Mean Absolute Error (MAE): 8.3239
Epoch 20/1000, Train MSE Loss: 113.8735
Dev Mean Absolute Error (MAE): 8.3202
Epoch 30/1000, Train MSE Loss: 110.3349
Dev Mean Absolute Error (MAE): 8.3200
Epoch 40/1000, Train MSE Loss: 113.5423
Dev Mean Absolute Error (MAE): 8.3167
Epoch 50/1000, Train MSE Loss: 112.7634
Dev Mean Absolute Error (MAE): 9.2524
Epoch 60/1000, Train MSE Loss: 116.3543
Dev Mean Absolute Error (MAE): 8.4285
Epoch 70/1000, Train MSE Loss: 110.1584
Dev Mean Absolute Error (MAE): 8.5342
Epoch 80/1000, Train MSE Loss: 118.1019
Dev Mean Absolute Error (MAE): 8.3223
Epoch 90/1000, Train MSE Loss: 117.0921
Dev Mean Absolute Error (MAE): 8.6142
Epoch 100/1000, Train MSE Loss: 112.3080
Dev Mean Absolute Error (MAE): 8.3307
Epoch 110/1000, Train MSE Loss: 109.7511
Dev Mean Absolute Error (MAE): 8.3446
Epoch 120/1000, Train MSE Loss: 110.9796
Dev Mean Absolute Error (MAE): 8.5353
Epoch 130/1000, Train MSE Loss: 109.8787
Dev Mean Absolute Er

#### 3.1 Analyze results on the dev set and prepare your submission file

Let's check our performance on the dev set:

In [18]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Predict the dev set
hyp, ref, files = predict_nn(model, slp_partitions['dev'])
filename = f'{DATADIR}/{trainset}/models/{model_id}/dev.pkl'
pickle.dump({'hyp':hyp, 'fileids':files}, open(filename, 'wb'))

# Report the results
print(f'Mean Absolute Error: {mean_absolute_error(ref, hyp):.2f}')
print(f'Mean Squared Error: {mean_squared_error(ref, hyp):.2f}')

# Predict the evl set
hyp, ref, files = predict_nn(model, slp_partitions['evl'])
filename = f'{DATADIR}/{trainset}/models/{model_id}/evl.pkl'
pickle.dump({'hyp':hyp, 'fileids':files}, open(filename, 'wb'))

Mean Absolute Error: 9.30
Mean Squared Error: 153.47


And generate the final prediction file and make a submission to the  [Kaggle competition](https://www.kaggle.com/t/8d80747e0c474688a83024aabdfe1ab0):

In [ ]:
from pf_tools import create_submission_file

students_group = '07' # <--- CHANGE THIS ACCORDINGLY

model_id_short = {
    'spkrec-xvect-voxceleb': 'nnet_spkrec-xvect',
    'spkrec-ecapa-voxceleb': 'nnet_spkrec-ecapa',
}.get(transform_id, f'nnet_{transform_id}')

results_path = f'{DATADIR}/{trainset}/models/{model_id}/'
filename = f'{CWD}/g{students_group}_{trainset}_{model_id_short}.csv' # <--- CHANGE THIS ACCORDINGLY

create_submission_file(results_path, filename)


At this point, you can explore different x-vector model configurations for feature extraction and alternative  neural model architectures and parameters.

### 3.2 Improving the neural network with standardized embeddings and early stopping

The baseline neural network above is intentionally simple. In this subsection we make the training procedure more robust by standardizing the embeddings, using a small hyperparameter grid, and early-stopping on MAE.

We will use an **age-binned stratified split** inside `train_small` so that the internal validation set has a similar age distribution to the fitting set. This is the regression analogue of stratified folds: we first bin the continuous ages into ranges, then stratify on those bins.


The next code cell defines a better MLP and the training utilities. Notice that we can still **evaluate** with MAE even if we **train** with another loss such as Huber. That is completely valid, and Huber is often a better optimization target when the final metric is MAE.


In [ ]:
# improvements pls

import copy
import pickle
import random
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import ParameterGrid, train_test_split
from sklearn.preprocessing import StandardScaler
from pf_tools import save_model

nn_device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {nn_device}')


def set_nn_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def make_age_bins(y, num_bins=5):
    y_series = pd.Series(np.asarray(y, dtype=np.float32))
    num_bins = min(num_bins, int(y_series.nunique()))
    if num_bins < 2:
        raise ValueError('Need at least two distinct ages to create age bins.')
    bins = pd.qcut(y_series, q=num_bins, duplicates='drop')
    return bins.cat.codes.to_numpy()


class ImprovedRegressor(nn.Module):
    def __init__(self, input_dim, hidden_dims=(128, 64), dropout=0.2):
        super().__init__()
        layers = []
        in_dim = input_dim

        for hidden_dim in hidden_dims:
            layers.append(nn.Linear(in_dim, hidden_dim))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            in_dim = hidden_dim

        layers.append(nn.Linear(in_dim, 1))
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)


def build_loss(loss_name):
    if loss_name == 'huber':
        return nn.HuberLoss(delta=1.0)
    if loss_name == 'mae':
        return nn.L1Loss()
    if loss_name == 'mse':
        return nn.MSELoss()
    raise ValueError(f'Unknown loss_name: {loss_name}')


def build_optimizer(config, model):
    if config['optimizer_name'] == 'adamw':
        return torch.optim.AdamW(
            model.parameters(),
            lr=config['lr'],
            weight_decay=config.get('weight_decay', 0.0),
        )

    if config['optimizer_name'] == 'sgd':
        return torch.optim.SGD(
            model.parameters(),
            lr=config['lr'],
            momentum=config.get('momentum', 0.9),
            weight_decay=config.get('weight_decay', 0.0),
        )

    raise ValueError(f"Unknown optimizer_name: {config['optimizer_name']}")


def make_tensor_loader(X, y, batch_size, shuffle):
    dataset = TensorDataset(
        torch.from_numpy(X.astype(np.float32)),
        torch.from_numpy(y.astype(np.float32)).view(-1, 1),
    )
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)


def predict_regressor(model, X, batch_size=128):
    dataset = TensorDataset(torch.from_numpy(X.astype(np.float32)))
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    predictions = []
    model.eval()
    with torch.no_grad():
        for (batch_X,) in loader:
            batch_pred = model(batch_X.to(nn_device)).squeeze(1).cpu().numpy()
            predictions.append(batch_pred)

    return np.concatenate(predictions)


def train_single_configuration(config, X_fit, y_fit, X_val, y_val):
    set_nn_seed(config.get('seed', 42))

    scaler = StandardScaler()
    X_fit_scaled = scaler.fit_transform(X_fit).astype(np.float32)
    X_val_scaled = scaler.transform(X_val).astype(np.float32)

    train_loader = make_tensor_loader(
        X_fit_scaled,
        y_fit,
        batch_size=config['batch_size'],
        shuffle=True,
    )

    model = ImprovedRegressor(
        input_dim=X_fit_scaled.shape[1],
        hidden_dims=config['hidden_dims'],
        dropout=config['dropout'],
    ).to(nn_device)

    criterion = build_loss(config['loss_name'])
    optimizer = build_optimizer(config, model)

    best_state = copy.deepcopy(model.state_dict())
    best_val_mae = float('inf')
    best_epoch = 0
    epochs_without_improvement = 0
    history = []

    for epoch in range(1, config['max_epochs'] + 1):
        model.train()
        total_loss = 0.0
        total_examples = 0

        for batch_X, batch_y in train_loader:
            batch_X = batch_X.to(nn_device)
            batch_y = batch_y.to(nn_device)

            optimizer.zero_grad()
            batch_pred = model(batch_X)
            loss = criterion(batch_pred, batch_y)
            loss.backward()

            grad_clip = config.get('grad_clip')
            if grad_clip is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)

            optimizer.step()

            total_loss += loss.item() * batch_X.size(0)
            total_examples += batch_X.size(0)

        val_pred = predict_regressor(model, X_val_scaled, batch_size=config['batch_size'])
        val_mae = mean_absolute_error(y_val, val_pred)
        train_loss = total_loss / max(total_examples, 1)
        history.append({'epoch': epoch, 'train_loss': train_loss, 'val_mae': val_mae})

        if val_mae < best_val_mae - 1e-6:
            best_val_mae = val_mae
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= config['patience']:
            break

    model.load_state_dict(best_state)

    return {
        'model': model,
        'scaler': scaler,
        'best_epoch': best_epoch,
        'val_mae': best_val_mae,
        'history': history,
    }



We now prepare the standardized-search experiment. The hyperparameter grid is intentionally moderate so that it is realistic to run on `train_small` on CPU.

The search varies:
1. optimizer (`AdamW` or `SGD`)
2. learning rate
3. weight decay
4. hidden-layer layout
5. dropout

We keep **Huber loss** fixed as the default training loss and still measure performance with MAE and MSE.


In [ ]:
# improvements pls

X_train_nn = data[trainset].astype(np.float32)
y_train_nn = labels_age[trainset].astype(np.float32)
X_dev_nn = data['dev'].astype(np.float32)
y_dev_nn = labels_age['dev'].astype(np.float32)
X_evl_nn = data['evl'].astype(np.float32)

age_bins = make_age_bins(y_train_nn, num_bins=5)
X_fit_nn, X_val_nn, y_fit_nn, y_val_nn = train_test_split(
    X_train_nn,
    y_train_nn,
    test_size=0.2,
    random_state=42,
    stratify=age_bins,
)

nn_search_spaces = [
    {
        'optimizer_name': ['adamw'],
        'lr': [1e-3, 3e-4],
        'weight_decay': [0.0, 1e-4],
        'hidden_dims': [(128,), (128, 64)],
        'dropout': [0.0, 0.2],
        'loss_name': ['huber'],
        'batch_size': [32],
        'max_epochs': [250],
        'patience': [25],
        'grad_clip': [1.0],
        'seed': [42],
    },
    {
        'optimizer_name': ['sgd'],
        'lr': [1e-2, 3e-3],
        'momentum': [0.9],
        'weight_decay': [0.0, 1e-4],
        'hidden_dims': [(128,), (128, 64)],
        'dropout': [0.0, 0.2],
        'loss_name': ['huber'],
        'batch_size': [32],
        'max_epochs': [250],
        'patience': [25],
        'grad_clip': [1.0],
        'seed': [42],
    },
]

nn_param_grid = []
for search_space in nn_search_spaces:
    nn_param_grid.extend(list(ParameterGrid(search_space)))

print(f'Internal fit split shape: {X_fit_nn.shape}')
print(f'Internal validation split shape: {X_val_nn.shape}')
print(f'Dev split shape: {X_dev_nn.shape}')
print(f'Total NN configurations to evaluate: {len(nn_param_grid)}')



The next cell runs the neural-network grid search. Each configuration is trained on the fitting split, early-stopped on the internal validation split, and then evaluated on the `dev` set for comparison.

The selected configuration is the one with the lowest **validation MAE**. That keeps the `dev` set closer to an external checkpoint instead of making it the optimization target.


In [ ]:
# improvements pls

nn_grid_results = []
best_nn_search_result = None

for idx, config in enumerate(nn_param_grid, start=1):
    print(f'[{idx}/{len(nn_param_grid)}] {config}')

    trained = train_single_configuration(
        config,
        X_fit_nn,
        y_fit_nn,
        X_val_nn,
        y_val_nn,
    )

    X_dev_scaled = trained['scaler'].transform(X_dev_nn).astype(np.float32)
    dev_pred = predict_regressor(
        trained['model'],
        X_dev_scaled,
        batch_size=config['batch_size'],
    )
    dev_mae = mean_absolute_error(y_dev_nn, dev_pred)
    dev_mse = mean_squared_error(y_dev_nn, dev_pred)

    result = {
        'config': config.copy(),
        'val_mae': trained['val_mae'],
        'dev_mae': dev_mae,
        'dev_mse': dev_mse,
        'best_epoch': trained['best_epoch'],
    }
    nn_grid_results.append(result)

    print(
        f'  best epoch: {result["best_epoch"]} | '
        f'val MAE: {result["val_mae"]:.3f} | '
        f'dev MAE: {result["dev_mae"]:.3f} | '
        f'dev MSE: {result["dev_mse"]:.3f}'
    )

    if best_nn_search_result is None or (
        result['val_mae'], result['dev_mae']
    ) < (
        best_nn_search_result['val_mae'], best_nn_search_result['dev_mae']
    ):
        best_nn_search_result = result

nn_grid_results = sorted(nn_grid_results, key=lambda result: (result['val_mae'], result['dev_mae']))

print() 
print('=== Improved NN grid results (sorted by validation MAE) ===')
print(f'{"Optimizer":<10}{"Hidden":<16}{"Drop":>8}{"LR":>10}{"Val MAE":>12}{"Dev MAE":>12}{"Dev MSE":>12}')
for result in nn_grid_results:
    cfg = result['config']
    print(
        f'{cfg["optimizer_name"]:<10}'
        f'{str(cfg["hidden_dims"]):<16}'
        f'{cfg["dropout"]:>8.2f}'
        f'{cfg["lr"]:>10.4g}'
        f'{result["val_mae"]:>12.3f}'
        f'{result["dev_mae"]:>12.3f}'
        f'{result["dev_mse"]:>12.3f}'
    )

print()
print('Selected configuration (lowest validation MAE):')
print(best_nn_search_result['config'])
print(
    f"Validation MAE = {best_nn_search_result['val_mae']:.3f} | "
    f"Dev MAE = {best_nn_search_result['dev_mae']:.3f}"
)



Once the best configuration is selected, we retrain it on the full `train_small` split and use the `dev` set for early stopping. This makes the final `dev` score slightly optimistic, but it usually gives a stronger model for generating the final `evl` predictions.


In [ ]:
# improvements pls

best_nn_config = best_nn_search_result['config'].copy()
print('Retraining the selected NN configuration on the full training split:')
print(best_nn_config)

final_nn_training = train_single_configuration(
    best_nn_config,
    X_train_nn,
    y_train_nn,
    X_dev_nn,
    y_dev_nn,
)

X_dev_scaled = final_nn_training['scaler'].transform(X_dev_nn).astype(np.float32)
X_evl_scaled = final_nn_training['scaler'].transform(X_evl_nn).astype(np.float32)

best_nn_dev_pred = predict_regressor(
    final_nn_training['model'],
    X_dev_scaled,
    batch_size=best_nn_config['batch_size'],
)
best_nn_evl_pred = predict_regressor(
    final_nn_training['model'],
    X_evl_scaled,
    batch_size=best_nn_config['batch_size'],
)

final_dev_mae = mean_absolute_error(y_dev_nn, best_nn_dev_pred)
final_dev_mse = mean_squared_error(y_dev_nn, best_nn_dev_pred)

print(f'Final dev MAE: {final_dev_mae:.3f}')
print(f'Final dev MSE: {final_dev_mse:.3f}')
print(f'Best early-stopping epoch on dev: {final_nn_training["best_epoch"]}')

nn_artifact = {
    'type': 'improved_nn',
    'model_class': 'ImprovedRegressor',
    'state_dict': {
        key: value.detach().cpu()
        for key, value in final_nn_training['model'].state_dict().items()
    },
    'input_dim': X_train_nn.shape[1],
    'hidden_dims': best_nn_config['hidden_dims'],
    'dropout': best_nn_config['dropout'],
    'scaler': final_nn_training['scaler'],
    'config': best_nn_config,
    'transform_id': transform_id,
    'audio_preprocess_id': audio_preprocess_id,
    'best_epoch': final_nn_training['best_epoch'],
}

model_id = save_model(nn_artifact, f'nnet_grid_{transform_id}', f'{DATADIR}/{trainset}/models/')
model_id_short = f'nnet-grid_{transform_id}'.replace('__', '_')

filename = f'{DATADIR}/{trainset}/models/{model_id}/dev.pkl'
pickle.dump({'hyp': best_nn_dev_pred, 'fileids': fileids['dev']}, open(filename, 'wb'))

filename = f'{DATADIR}/{trainset}/models/{model_id}/evl.pkl'
pickle.dump({'hyp': best_nn_evl_pred, 'fileids': fileids['evl']}, open(filename, 'wb'))

print(f'Improved NN artifact saved as {model_id}')
print(f'Dev/Evl prediction files saved under {DATADIR}/{trainset}/models/{model_id}/')



The last cell mirrors the earlier submission workflow, but it uses the improved neural-network model selected in this subsection.


In [ ]:
# improvements pls

from pf_tools import create_submission_file

students_group = '07'  # <--- CHANGE THIS ACCORDINGLY

results_path = f'{DATADIR}/{trainset}/models/{model_id}/'
filename = f'{CWD}/g{students_group}_{trainset}_{model_id_short}.csv'

create_submission_file(results_path, filename)



In [ ]:
# improvements pls

import pickle
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from pf_tools import save_model

trainset = 'train_small'  # change to 'train' after extracting the full training split

X_train = data[trainset]
y_train = labels_age[trainset].astype(np.float32)
X_dev = data['dev']
y_dev = labels_age['dev'].astype(np.float32)
X_evl = data['evl']

cv = KFold(n_splits=5, shuffle=True, random_state=42)


def run_grid(name, pipe, param_grid, cv=cv):
    grid = GridSearchCV(
        pipe,
        param_grid=param_grid,
        scoring='neg_mean_absolute_error',
        cv=cv,
        n_jobs=-1,
        verbose=1,
        refit=True,
    )
    grid.fit(X_train, y_train)

    dev_pred = grid.predict(X_dev)
    dev_mae = mean_absolute_error(y_dev, dev_pred)
    dev_mse = mean_squared_error(y_dev, dev_pred)

    print(f'\n=== {name} ===')
    print(f'Best params:    {grid.best_params_}')
    print(f'CV MAE (train): {-grid.best_score_:.3f}')
    print(f'Dev MAE:        {dev_mae:.3f}')
    print(f'Dev MSE:        {dev_mse:.3f}')

    return {
        'name': name,
        'grid': grid,
        'dev_pred': dev_pred,
        'dev_mae': dev_mae,
        'dev_mse': dev_mse,
    }


In [22]:
# improvements pls

improvement_results = []

pipe_ridge = Pipeline([
    ('scaler', StandardScaler()),
    ('ridge', Ridge()),
])
grid_ridge = {
    'ridge__alpha': [0.01, 0.1, 1, 10, 100],
}
improvement_results.append(run_grid('Ridge', pipe_ridge, grid_ridge))

pipe_lin = Pipeline([
    ('scaler', StandardScaler()),
    ('svr', SVR(kernel='linear')),
])
grid_lin = {
    'svr__C': [0.01, 0.1, 1, 10, 100],
    'svr__epsilon': [0.1, 0.5, 1.0, 2.0],
}
improvement_results.append(run_grid('Linear SVR', pipe_lin, grid_lin))

pipe_rbf = Pipeline([
    ('scaler', StandardScaler()),
    ('svr', SVR(kernel='rbf')),
])
grid_rbf = {
    'svr__C': [0.1, 1, 10, 100, 1000],
    'svr__gamma': ['scale', 1e-4, 1e-3, 1e-2, 1e-1],
    'svr__epsilon': [0.1, 0.5, 1.0, 2.0],
}
improvement_results.append(run_grid('RBF SVR', pipe_rbf, grid_rbf))

pipe_gbr = Pipeline([
    ('gbr', GradientBoostingRegressor(random_state=42)),
])
grid_gbr = {
    'gbr__n_estimators': [100, 300],
    'gbr__learning_rate': [0.05, 0.1],
    'gbr__max_depth': [3, 5],
    'gbr__subsample': [0.8, 1.0],
}
improvement_results.append(run_grid('Gradient Boosting', pipe_gbr, grid_gbr))

improvement_results = sorted(improvement_results, key=lambda r: r['dev_mae'])

print('\n=== Results (sorted by dev MAE) ===')
print(f'{"Model":<22}{"dev MAE":>10}{"dev MSE":>10}')
for result in improvement_results:
    print(f'{result["name"]:<22}{result["dev_mae"]:>10.3f}{result["dev_mse"]:>10.3f}')

best_result = improvement_results[0]
best_name = best_result['name']
best_grid = best_result['grid']
print(f'\nBest model: {best_name}  |  dev MAE = {best_result["dev_mae"]:.3f}')


Fitting 5 folds for each of 5 candidates, totalling 25 fits

=== Ridge ===
Best params:    {'ridge__alpha': 100}
CV MAE (train): 7.102
Dev MAE:        6.805
Dev MSE:        67.214
Fitting 5 folds for each of 20 candidates, totalling 100 fits

=== Linear SVR ===
Best params:    {'svr__C': 0.01, 'svr__epsilon': 2.0}
CV MAE (train): 7.140
Dev MAE:        7.715
Dev MSE:        81.530
Fitting 5 folds for each of 100 candidates, totalling 500 fits

=== RBF SVR ===
Best params:    {'svr__C': 100, 'svr__epsilon': 2.0, 'svr__gamma': 0.0001}
CV MAE (train): 7.074
Dev MAE:        7.336
Dev MSE:        71.904
Fitting 5 folds for each of 16 candidates, totalling 80 fits

=== Gradient Boosting ===
Best params:    {'gbr__learning_rate': 0.05, 'gbr__max_depth': 3, 'gbr__n_estimators': 300, 'gbr__subsample': 0.8}
CV MAE (train): 7.466
Dev MAE:        7.615
Dev MSE:        90.640

=== Results (sorted by dev MAE) ===
Model                    dev MAE   dev MSE
Ridge                      6.805    67.214
RB

In [ ]:
# improvements pls

best_estimator = best_grid.best_estimator_
best_dev_pred = best_estimator.predict(X_dev)
best_evl_pred = best_estimator.predict(X_evl)

model_id = save_model(best_estimator, f'grid_{transform_id}', f'{DATADIR}/{trainset}/models/')
model_id_short = 'grid-search-xvec'

filename = f'{DATADIR}/{trainset}/models/{model_id}/dev.pkl'
pickle.dump({'hyp': best_dev_pred, 'fileids': fileids['dev']}, open(filename, 'wb'))

filename = f'{DATADIR}/{trainset}/models/{model_id}/evl.pkl'
pickle.dump({'hyp': best_evl_pred, 'fileids': fileids['evl']}, open(filename, 'wb'))

print(f'Best grid-search model saved as {model_id}')
print(f'Dev/Evl predictions saved under {DATADIR}/{trainset}/models/{model_id}/')


# Contacts and support
You can contact the professors during the classes or the office hours.

Particularly, for this second laboratory assignment, you should contact Prof. Alberto Abad: alberto.abad@tecnico.ulisboa.pt


